In [11]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vinayak121_aluminum_datasets_path = kagglehub.dataset_download('vinayak121/aluminum-datasets')

print('Data source import complete.')


ModuleNotFoundError: No module named 'kagglehub'

In [12]:
# Robust preprocessing and sanitization
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# Reload raw data and fill NaNs before scaling
raw_df = pd.read_csv('final_aluminum_data.csv')

# Ensure required columns exist
composition_cols = ['Al', 'Cu', 'Mg', 'Mn', 'Si', 'Zn', 'Fe', 'Ni', 'Cr', 'Ti', 'Pb', 'Sn', 'Zr', 'Co', 'V']
processing_cols = ['solution_temp', 'solution_time', 'aging_temp', 'aging_time', 'strain_hardening_index']
input_features = composition_cols + processing_cols

missing_cols = [c for c in input_features if c not in raw_df.columns]
if len(missing_cols) > 0:
    raise ValueError(f"Missing expected columns in final_aluminum_data.csv: {missing_cols}")

# Fill NaNs and clip extreme values for stability
stable_df = raw_df.copy()
stable_df[input_features] = stable_df[input_features].replace([np.inf, -np.inf], np.nan)
stable_df[input_features] = stable_df[input_features].fillna(0.0)

# Optional: clip physically invalid negatives in processing features
for c in processing_cols:
    stable_df[c] = np.clip(stable_df[c].values, a_min=0.0, a_max=None)

# Dataset with explicit nan/inf guards
class AlloyDataset(Dataset):
    def __init__(self, data: pd.DataFrame, input_features: list[str]):
        self.features = input_features
        self.scaler = StandardScaler()
        x_np = data[self.features].to_numpy(dtype=np.float64)
        # Replace any residual NaN/Inf
        x_np = np.nan_to_num(x_np, nan=0.0, posinf=0.0, neginf=0.0)
        x_scaled = self.scaler.fit_transform(x_np)
        x_scaled = np.nan_to_num(x_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        self.X = torch.from_numpy(x_scaled).float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]
        # Final guard
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        return x

    def get_scaler(self):
        return self.scaler

# Build dataset and basic checks
dataset = AlloyDataset(stable_df, input_features)
print(f"Dataset size (stable): {len(dataset)}")
print(f"Input dimension (stable): {dataset[0].shape[0]}")

# Sanity checks
with torch.no_grad():
    x_sample = dataset[0]
    assert torch.isfinite(x_sample).all(), "Found non-finite values in dataset sample"
    x_all = dataset.X
    print("Finite check ->", torch.isfinite(x_all).all().item())
    print("Abs mean/std ->", x_all.abs().mean().item(), x_all.std().item())


Dataset size (stable): 1154
Input dimension (stable): 20
Finite check -> True
Abs mean/std -> 0.5897054076194763 1.0000216960906982


In [13]:
# Stable loss and training utilities
import torch.nn as nn
import torch.optim as optim

# Numerically stable VAE loss
def loss_function_stable(recon_x, x, mu, log_var):
    # Reconstruction loss
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    # Clamp log_var to avoid extreme values
    log_var = torch.clamp(log_var, min=-30.0, max=20.0)
    # Compute KL divergence safely: 0.5 * sum(exp(log_var) + mu^2 - 1 - log_var)
    kl_per_dim = torch.exp(log_var) + mu.pow(2) - 1.0 - log_var
    kl_loss = 0.5 * torch.mean(kl_per_dim)
    total = recon_loss + kl_loss
    if not torch.isfinite(total):
        # Guard against NaNs/Infs propagating
        total = torch.nan_to_num(total, nan=0.0, posinf=1e6, neginf=1e6)
    return total, recon_loss, kl_loss

# Hardened training epoch
def train_epoch_hardened(model, dataloader, optimizer, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    n_batches = 0

    for batch in dataloader:
        batch = torch.nan_to_num(batch.to(device), nan=0.0, posinf=0.0, neginf=0.0)
        optimizer.zero_grad(set_to_none=True)

        recon_batch, mu, log_var = model(batch)
        # Guards on model outputs
        recon_batch = torch.nan_to_num(recon_batch, nan=0.0, posinf=0.0, neginf=0.0)
        mu = torch.nan_to_num(mu, nan=0.0, posinf=0.0, neginf=0.0)
        log_var = torch.nan_to_num(log_var, nan=0.0, posinf=0.0, neginf=0.0)

        loss, recon_loss, kl_loss = loss_function_stable(recon_batch, batch, mu, log_var)
        loss.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        total_loss += float(loss.detach().cpu())
        total_recon += float(recon_loss.detach().cpu())
        total_kl += float(kl_loss.detach().cpu())
        n_batches += 1

    return total_loss / n_batches, total_recon / n_batches, total_kl / n_batches

# Hardened validation
@torch.no_grad()
def validate_hardened(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    n_batches = 0

    for batch in dataloader:
        batch = torch.nan_to_num(batch.to(device), nan=0.0, posinf=0.0, neginf=0.0)
        recon_batch, mu, log_var = model(batch)
        recon_batch = torch.nan_to_num(recon_batch, nan=0.0, posinf=0.0, neginf=0.0)
        mu = torch.nan_to_num(mu, nan=0.0, posinf=0.0, neginf=0.0)
        log_var = torch.nan_to_num(log_var, nan=0.0, posinf=0.0, neginf=0.0)

        loss, recon_loss, kl_loss = loss_function_stable(recon_batch, batch, mu, log_var)
        total_loss += float(loss.detach().cpu())
        total_recon += float(recon_loss.detach().cpu())
        total_kl += float(kl_loss.detach().cpu())
        n_batches += 1

    return total_loss / n_batches, total_recon / n_batches, total_kl / n_batches


In [14]:
# K-fold training using hardened loops
from torch.utils.data import DataLoader, SubsetRandomSampler
from sklearn.model_selection import KFold

# Reuse existing CVAE definition from the notebook (assumed defined earlier)

@torch.no_grad()
def count_nonfinite_tensors(model):
    bad = 0
    for p in model.parameters():
        if p.grad is not None:
            if not torch.isfinite(p.grad).all():
                bad += 1
    return bad


def train_kfold_hardened(dataset, input_dim, hidden_dims=[128,64,32], latent_dim=12,
                         n_splits=5, epochs=100, batch_size=32, learning_rate=1e-4, weight_decay=0.0):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    histories = []
    models = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\nTraining Fold {fold + 1}/{n_splits}")

        train_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
        val_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))

        model = CVAE(input_dim=input_dim, hidden_dims=hidden_dims, latent_dim=latent_dim).to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        history = {
            'train_loss': [], 'train_recon_loss': [], 'train_kl_loss': [],
            'val_loss': [], 'val_recon_loss': [], 'val_kl_loss': []
        }

        for epoch in range(epochs):
            train_loss, train_recon, train_kl = train_epoch_hardened(model, train_loader, optimizer)
            val_loss, val_recon, val_kl = validate_hardened(model, val_loader)

            history['train_loss'].append(train_loss)
            history['train_recon_loss'].append(train_recon)
            history['train_kl_loss'].append(train_kl)
            history['val_loss'].append(val_loss)
            history['val_recon_loss'].append(val_recon)
            history['val_kl_loss'].append(val_kl)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] - Train: {train_loss:.6f} (recon {train_recon:.6f}, kl {train_kl:.6f}) | "
                      f"Val: {val_loss:.6f} (recon {val_recon:.6f}, kl {val_kl:.6f})")

        histories.append(history)
        models.append(model)

    return models, histories

print("Starting hardened K-fold training...")
models, histories = train_kfold_hardened(dataset, input_dim=len(input_features),
                                         hidden_dims=[128,64,32], latent_dim=12,
                                         n_splits=5, epochs=100, batch_size=32, learning_rate=1e-4,
                                         weight_decay=1e-5)


Starting hardened K-fold training...

Training Fold 1/5
Epoch [10/100] - Train: 1.135026 (recon 1.009421, kl 0.125605) | Val: 1.244655 (recon 1.032456, kl 0.212200)
Epoch [20/100] - Train: 1.019581 (recon 0.910287, kl 0.109293) | Val: 1.066974 (recon 0.941201, kl 0.125773)
Epoch [30/100] - Train: 0.975448 (recon 0.860829, kl 0.114619) | Val: 1.422240 (recon 1.273283, kl 0.148957)
Epoch [40/100] - Train: 0.921729 (recon 0.798656, kl 0.123073) | Val: 0.962889 (recon 0.825400, kl 0.137490)
Epoch [50/100] - Train: 0.905704 (recon 0.774109, kl 0.131595) | Val: 1.479232 (recon 1.310899, kl 0.168333)
Epoch [60/100] - Train: 0.883503 (recon 0.743165, kl 0.140339) | Val: 0.924713 (recon 0.751089, kl 0.173624)
Epoch [70/100] - Train: 0.882541 (recon 0.734240, kl 0.148301) | Val: 0.899787 (recon 0.750775, kl 0.149012)
Epoch [80/100] - Train: 0.862006 (recon 0.704626, kl 0.157380) | Val: 0.882201 (recon 0.709787, kl 0.172414)
Epoch [90/100] - Train: 0.832751 (recon 0.668177, kl 0.164574) | Val: 0.

In [15]:
# Safe generation utilities
import pandas as pd

@torch.no_grad()
def generate_compositions_safe(model, scaler, n_samples=100):
    model.eval()
    z = torch.randn(n_samples, model.latent_dim, device=device)
    decoded = model.decoder(z)
    decoded = torch.nan_to_num(decoded, nan=0.0, posinf=0.0, neginf=0.0)
    samples = decoded.detach().cpu().numpy()
    # Inverse transform
    samples = scaler.inverse_transform(samples)
    gen_df = pd.DataFrame(samples, columns=input_features)
    return gen_df

# Validate and normalize composition fractions

def validate_and_normalize_compositions(df_in: pd.DataFrame, composition_cols: list[str]) -> pd.DataFrame:
    comps = df_in[composition_cols].copy()
    comps = comps.fillna(0.0)
    comps[comps < 0] = 0.0
    row_sums = comps.sum(axis=1).replace(0.0, 1.0)
    comps = comps.div(row_sums, axis=0) * 100.0
    out = df_in.copy()
    for c in composition_cols:
        out[c] = comps[c].values
    return out

# Choose best model by last validation loss
val_losses = [h['val_loss'][-1] for h in histories]
best_idx = int(np.argmin(val_losses))
best_model = models[best_idx]
print(f"Using best model from fold {best_idx+1}")

# Generate, validate, and save
n_samples = 200
new_comp_df = generate_compositions_safe(best_model, dataset.get_scaler(), n_samples)
new_comp_df = validate_and_normalize_compositions(new_comp_df, composition_cols)

print("Generated composition sums (should be ~100):")
print(new_comp_df[composition_cols].sum(axis=1).describe())

new_comp_df.to_csv('new_compositions.csv', index=False)
print("Saved new compositions -> new_compositions.csv")


Using best model from fold 2
Generated composition sums (should be ~100):
count    200.000000
mean     100.000000
std        0.000012
min       99.999969
25%       99.999992
50%      100.000000
75%      100.000008
max      100.000031
dtype: float64
Saved new compositions -> new_compositions.csv
